# Inventory Sentinel — 3-Level MARL + Flower Federation Training

Trains the L1 (per-store), L2 (per-zone), L3 (per-city) hierarchy and runs
Flower federated rounds across stores so raw demand never centralizes (I-11, DPDPA-A).

**Reward (independent — I-2)**: per-tier formulation; L1 maximizes service-level subject to inventory cost cap; L2 minimizes cross-store stockouts; L3 balances city-wide carbon impact.

In [ ]:
%pip install --quiet torch flwr ray[rllib] mlflow pyro-ppl structlog

In [ ]:
import os, random, numpy as np, torch, mlflow
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
mlflow.set_tracking_uri(os.environ.get('MLFLOW_TRACKING_URI', 'http://localhost:5000'))
CITY = os.environ.get('CITY', 'bengaluru')
mlflow.set_experiment(f'{CITY}_inventory_sentinel' if CITY=='bengaluru' else f'mumbai_transfer_inventory_sentinel_l1')

In [ ]:
import sys; sys.path.append('.')
from agents.inventory_sentinel.training.train import train_local_l1
with mlflow.start_run() as run:
    metrics = train_local_l1(city=CITY, episodes=200, seed=SEED, device=DEVICE)
    mlflow.log_metrics(metrics)
    print(metrics)

In [ ]:
# Flower federated round (gradient-only aggregation — DPDPA-A compliance)
from agents.inventory_sentinel.training.flower_client import start_client
from agents.inventory_sentinel.training.flower_server import start_server_in_thread
server_thread = start_server_in_thread(num_rounds=3)
start_client(client_id='store-001', server_address='localhost:8080')
server_thread.join(timeout=120)